# Create Sweden-America Foundation Awards (FELLOWSHIP PATTERN, sitemap-driven WP scrape)

Sweden-America Foundation (Sverige-Amerika Stiftelsen, founded 1919) sends
Swedish graduate students and post-docs to U.S. host institutions on annual
fellowships. The foundation publishes per-fellow "travel reports" on its
WordPress site at sweamfo.se under `/reserapporter-fran-stipendiater/{slug}/`.
Each page header follows a clean canonical pattern:

    <h1>Stipendiat YYYY-YYYY: Given Family</h1>

which makes name + fellowship-year-range trivial to extract. Body text
names the host institution (Swedish: `vid <Institution>`) and gives a
first-person Swedish summary of the postdoctoral / graduate work done at
that institution.

**Awarding body:** Sweden-America Foundation - F4320320938 (SE)

**Source:** https://sweamfo.se/wp-sitemap-posts-page-1.xml enumerates 46 fellow profile URLs (2026-05-27 snapshot); each profile is scraped for the h1 header + body-text institution.

**Schema choices:**
- One row per fellow. `funder_award_id` = `sweden-america-{slug}` (WP post slug).
- `funder_scheme` = `Sweden-America Foundation Fellowship` (single program).
- `funding_type` = `fellowship`.
- `start_year` / `end_year` from the YYYY-YYYY range in the header (covers the academic year).
- `lead_investigator.affiliation.country` = `US` hardcoded — the foundation's mission is exclusively Swedish-fellow placement at U.S. host institutions.
- `amount` / `currency` ship NULL with **§6.7 amount waiver**. The foundation publishes a general range on its eligibility page (typical SEK 100,000-250,000) but per-fellow allocations are not disclosed on profile pages. Same waiver pattern as HHMI #44 / Damon Runyon #73 / CIFAR #79 / Searle Scholars #133.
- Coverage caveat: only 46 of the documented ~3000 historical Sweden-America fellows are exposed on the live site (most as 2020-2024 modern recipients with travel reports + a handful of historical name-only stubs including Bertil Ohlin, 1922-23 fellow and 1977 Nobel laureate). The 2,950+ pre-2020 fellows aren't on the live site — documented in tracker as a Step 0 follow-up.

**S3 location:** `s3a://openalex-ingest/awards/sweden_america_fdn/sweden_america_fdn_fellows.parquet`

**Method:** sitemap-enumerated WP page scrape — same idiom as Damon Runyon (#73, Drupal sitemap) and Sweden-America's own well-defined h1 makes name + year extraction more deterministic than the body-text-only foundations.

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sweden_america_fdn_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/sweden_america_fdn/sweden_america_fdn_fellows.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.sweden_america_fdn_raw;

## Step 1.5: Inspect raw + year/institution distribution

Expected: ~46 fellows, year range 1922-2024 (mostly 2020-2024 modern cohort), 100% on name + description + start_year, ~89% on institution (5 of 46 have free-text institutions that don't match the `vid <Institution>` Swedish-language anchor).

In [ ]:
%sql
DESCRIBE openalex.awards.sweden_america_fdn_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.sweden_america_fdn_raw LIMIT 5;

In [ ]:
%sql
-- Year range + per-year counts.
SELECT start_year, COUNT(*) AS rows,
       COUNT(institution) AS has_institution
FROM openalex.awards.sweden_america_fdn_raw
GROUP BY start_year
ORDER BY start_year;

In [ ]:
%sql
-- Top host institutions. Expect Columbia, Harvard, Cornell, Stanford dominating.
SELECT institution, COUNT(*) AS rows
FROM openalex.awards.sweden_america_fdn_raw
WHERE institution IS NOT NULL
GROUP BY institution
ORDER BY rows DESC
LIMIT 15;

## Step 1.6: Fail-fast - verify Sweden-America funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320320938;

## Step 2: Transform to award schema

Per-row `display_name` = `Sweden-America Foundation Fellowship YYYY-YYYY: Given Family`. `description` is the first ~500 chars of the Swedish travel-report body. `lead_investigator.affiliation.country` = `'US'` (the host country, per the foundation's exclusively-US placement program).

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sweden_america_fdn_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320938  -- Sweden-America Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT('Sweden-America Foundation Fellowship ',
           n.start_year, '-', n.end_year, ': ', n.name) AS display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,
    CAST(NULL AS STRING) AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'fellowship' AS funding_type,
    'Sweden-America Foundation Fellowship' AS funder_scheme,
    'sweden_america_foundation' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.start_year AS INT) AS start_year,
    TRY_CAST(n.end_year AS INT) AS end_year,
    CASE
        WHEN n.name IS NULL OR n.name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.institution AS name,
                CAST('US' AS STRING) AS country,  -- Sweden-America places fellows exclusively at US institutions
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.sweden_america_fdn_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 145

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'sweden_america_foundation' AND priority = 145;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    145 AS priority  -- Sweden-America Foundation
FROM openalex.awards.sweden_america_fdn_awards;

## Step 6: Verification

§6.7 amount-coverage check **WAIVED** (same justification as HHMI #44 / Damon Runyon #73 / CIFAR #79 / Searle Scholars #133). Expect ~46 rows, 100% on title + description + start_year + funder, ~89% on institution, 0% amount.

In [ ]:
%sql
SELECT COUNT(*) AS total_sweden_america_rows FROM openalex.awards.sweden_america_fdn_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.sweden_america_fdn_awards;

In [ ]:
%sql
-- §6.3 Data completeness. pct_amount expected 0 (waived).
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_pi,
    COUNT(start_year) AS has_start_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.sweden_america_fdn_awards;

In [ ]:
%sql
-- §6.7 waiver confirmation. Expect all NULL.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.sweden_america_fdn_awards;

In [ ]:
%sql
-- Year-range sanity. Expect 1922-2024.
SELECT MIN(start_year) AS min_year, MAX(start_year) AS max_year,
       COUNT(DISTINCT start_year) AS distinct_years
FROM openalex.awards.sweden_america_fdn_awards;

In [ ]:
%sql
-- Country sanity — should be 100% US.
SELECT lead_investigator.affiliation.country AS country, COUNT(*) AS rows
FROM openalex.awards.sweden_america_fdn_awards
GROUP BY country
ORDER BY rows DESC;

In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.sweden_america_fdn_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample 10 fellows for eyeball QA.
SELECT id, SUBSTRING(display_name, 1, 70) AS title, start_year, end_year,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 45) AS institution
FROM openalex.awards.sweden_america_fdn_awards
ORDER BY start_year DESC, family
LIMIT 10;